# NCAA March Madness Modeling & Prediction (Combined)

This notebook handles:
1. Creating a matchup-level training dataset from combined team features.
2. Training a predictive model (XGBoost).
3. Evaluating performance via Log Loss.
4. Generating predictions for the tournament (Men & Women).

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import log_loss
import os

DATA_PATH = './march-machine-learning-mania-2026/'

# Load precomputed features and tournament results (Both M & W)
team_features = pd.read_csv('all_team_features.csv')
m_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'MNCAATourneyDetailedResults.csv'))
w_tourney_results = pd.read_csv(os.path.join(DATA_PATH, 'WNCAATourneyDetailedResults.csv'))

# Combine tournament results for training
tourney_results = pd.concat([m_tourney_results, w_tourney_results], ignore_index=True)

print("Data loaded.")

## 1. Constructing the Training Dataset

In [ ]:
def prepare_training_data(results_df, features_df):
    training_rows = []
    
    # Pre-index features for speed
    feat_dict = features_df.set_index(['Season', 'TeamID']).to_dict('index')
    
    for i, row in results_df.iterrows():
        season = row['Season']
        t1 = row['WTeamID'] if row['WTeamID'] < row['LTeamID'] else row['LTeamID']
        t2 = row['LTeamID'] if row['WTeamID'] < row['LTeamID'] else row['WTeamID']
        
        label = 1 if row['WTeamID'] == t1 else 0
        
        try:
            f1 = feat_dict[(season, t1)]
            f2 = feat_dict[(season, t2)]
            
            diff_features = {
                'Season': season,
                'SeedDiff': f1['SeedInt'] - f2['SeedInt'],
                'OffEffDiff': f1['OffEff'] - f2['OffEff'],
                'RankDiff': f1['AvgRank'] - f2['AvgRank'],
                'eFGDiff': f1['eFG'] - f2['eFG'],
                'ScoreDiff': f1['Score'] - f2['Score'],
                'label': label
            }
            training_rows.append(diff_features)
        except KeyError:
            continue # Skip games where features are missing
        
    return pd.DataFrame(training_rows)

train_df = prepare_training_data(tourney_results, team_features)
print(f"Training dataset created with {len(train_df)} games.")

## 2. Model Training

In [ ]:
X = train_df.drop(['Season', 'label'], axis=1)
y = train_df['label']

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = xgb.XGBClassifier(eval_metric='logloss')
    model.fit(X_train, y_train)
    
    preds = model.predict_proba(X_val)[:, 1]
    score = log_loss(y_val, preds)
    scores.append(score)

print(f"Average CV Log Loss: {np.mean(scores):.4f}")

## 3. Generate Submission

In [ ]:
submission = pd.read_csv(os.path.join(DATA_PATH, 'SampleSubmissionStage1.csv'))

final_model = xgb.XGBClassifier(eval_metric='logloss')
final_model.fit(X, y)

feat_dict = team_features.set_index(['Season', 'TeamID']).to_dict('index')

def get_submission_pred(row):
    parts = row['ID'].split('_')
    season, t1, t2 = int(parts[0]), int(parts[1]), int(parts[2])
    
    try:
        f1 = feat_dict[(season, t1)]
        f2 = feat_dict[(season, t2)]
        
        features = pd.DataFrame([{
            'SeedDiff': f1['SeedInt'] - f2['SeedInt'],
            'OffEffDiff': f1['OffEff'] - f2['OffEff'],
            'RankDiff': f1['AvgRank'] - f2['AvgRank'],
            'eFGDiff': f1['eFG'] - f2['eFG'],
            'ScoreDiff': f1['Score'] - f2['Score']
        }])
        return final_model.predict_proba(features)[:, 1][0]
    except KeyError:
        return 0.5 # Baseline for missing data

submission['Pred'] = submission.apply(get_submission_pred, axis=1)
submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' generated successfully!")